In [ ]:
import os
from dotenv import load_dotenv

from langchain_gigachat.chat_models import GigaChat
from langchain_core.prompts import (
    PromptTemplate,
    ChatPromptTemplate,
    SystemMessagePromptTemplate,
    HumanMessagePromptTemplate,
)
from langchain_core.output_parsers import StrOutputParser

load_dotenv()

giga_key = os.getenv("GIGA_KEY")
if not giga_key:
    raise ValueError("Ключ GIGA_KEY не найден в .env")

In [ ]:
llm = GigaChat(
    credentials=giga_key,
    model="GigaChat-2",
    verify_ssl_certs=False,
    scope="GIGACHAT_API_PERS",
    temperature=0.2,
    max_tokens=1000
)

In [ ]:
response = llm.invoke("Привет! Как дела?")
print(response.content)

In [ ]:
basic_prompt = PromptTemplate(
    input_variables=["text"],
    template="""
Проанализируй следующий текст заявки на аренду жилья и извлеки количество человек, которые будут проживать.
Текст заявки: {text}
Верни только число (целое число), соответствующее количеству проживающих.
Если количество не указано явно, постарайся определить его по контексту.
Количество человек:"""
)

chain = basic_prompt | llm | StrOutputParser()

In [ ]:
test_texts = [
    "Ищу квартиру для семьи из четырех человек на длительный срок",
    "Нужна студия для проживания одного человека рядом с метро",
    "Требуется двухкомнатная квартира для молодой пары",
    "Снимем жилье для троих студентов на учебный год",
    "Семья с двумя детьми ищет просторную квартиру",
    "Сниму недорогое жилье на 6 чел 3 взр и 3 дет 30.07-09.08",
    "Здравствуйте. Снимем жилье с 12.07 по 17.07. 2 взрослых и 1 ребенок 6 лет, со всеми удобствами.",
    "Добрый вечер! Ищем жилье на 2 человек, с 3 сентября на 7 дней.",
    "Ищем жильё на 4 взрослых с 31.08 по 07.09, желательно центр",
    "Срочно ищем жилье с 13.06 по 20.06 нас 3 человека(все взрослые). Желательно с удобствами в номере.",
    "Ищем жилье для двоих, 2-местный номер с удобствами, с 7 сентября не менее чем на неделю",
    "Ищем жильё 2 взрослых и ребёнок 6 лет. С 8 июля по 21 июля",
    "Ищем жильё 4 чел. 2 взрослых и 2 детей, сумма не более 65000 р. Квартира 2-х ком.",
    "Интересует жилье на одного человека с 08.07. до 15.07. Не далеко от пляжа",
    "Сниму жильё на 2 недели. Конец августа. Два взрослых человека. По адекватной цене."
]

for text in test_texts:
    result = chain.invoke({"text": text})
    print(f"Текст: {text}")
    print(f"Результат: {result}")
    print("---")

In [ ]:
import pandas as pd

df = pd.read_csv('rental_29.csv', sep=';')
df.head()

In [ ]:
results = []

for _, row in df.iterrows():
    text = row["text"]
    try:
        result = chain.invoke({"text": text})
        results.append(result.strip())
    except Exception as e:
        results.append(f"ERROR: {e}")

df["result"] = results
df.to_csv("rental_29_with_results.csv", index=False, encoding="utf-8-sig")
print("Готово!")
df

In [ ]:
df["result_num"] = pd.to_numeric(df["result"], errors="coerce")
df[["amount", "result", "result_num"]].head(15)

In [ ]:
correct = (df["amount"] == df["result_num"]).sum()
total = len(df)
errors = total - correct
accuracy = correct / total

print(f"Верных ответов: {correct}")
print(f"Ошибок: {errors}")
print(f"Точность: {accuracy:.1%}")

In [ ]:
mistakes = df[df["amount"] != df["result_num"]][["amount", "text", "result"]]
mistakes